# 03 — Speed + Memory Benchmarks

Measure tokens/sec and peak memory per parser on both languages. Three repeats, averaged.

In [1]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    # Kaggle ships pandas built against numpy 2.x, but spacy 3.7.5 pins numpy<2.
    # Force-reinstall pandas+numpy together so binary ABI matches.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps",
                    "numpy<2", "pandas>=2.0,<2.3"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


Local env detected — skipping cloud setup.


In [2]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import statistics
import pandas as pd

from src.data import load_sentences
from src.parsers import SpacyParser, StanzaParser
from src.perf import measure_throughput, measure_peak_memory

EN_TOKS = [s.tokens for s in load_sentences(Path("../data/en_ewt_test.conllu"))]
RU_TOKS = [s.tokens for s in load_sentences(Path("../data/ru_syntagrus_test.conllu"))]
print(f"EN: {len(EN_TOKS)} sents | RU: {len(RU_TOKS)} sents")

EN: 2077 sents | RU: 8800 sents


In [3]:
BATCH_SIZE = 32
REPEATS = 3
WARMUP = 3

def make_batches(tokens, batch_size=BATCH_SIZE):
    return [tokens[i:i+batch_size] for i in range(0, len(tokens), batch_size)]

def bench(parser, tokens):
    batches = make_batches(tokens)
    speeds = []
    for _ in range(REPEATS):
        t = measure_throughput(parser.parse, batches, warmup=WARMUP)
        speeds.append(t.tokens_per_sec)
    mean_tps = statistics.mean(speeds)
    std_tps = statistics.stdev(speeds) if REPEATS > 1 else 0.0
    peak = measure_peak_memory(lambda: parser.parse(tokens[:200]))
    return mean_tps, std_tps, peak

configs = [
    ("en", EN_TOKS, SpacyParser("en_core_web_trf")),
    ("en", EN_TOKS, StanzaParser("en")),
    ("ru", RU_TOKS, SpacyParser("ru_core_news_lg")),
    ("ru", RU_TOKS, StanzaParser("ru")),
]

rows = []
for lang, toks, parser in configs:
    print(f"Benchmarking {lang} {parser.name} ...")
    mean_tps, std_tps, peak = bench(parser, toks)
    rows.append({
        "lang": lang, "parser": parser.name,
        "family": "transition" if "spacy" in parser.name else "graph",
        "tokens_per_sec_mean": round(mean_tps, 1),
        "tokens_per_sec_std": round(std_tps, 1),
        "peak_mb": round(peak / 1e6, 1),
    })
    print(f"  {mean_tps:.0f} tok/s ±{std_tps:.0f}  peak={peak/1e6:.1f} MB")

Benchmarking en spacy:en_core_web_trf ...


/Users/aleksandrgavkovskij/programming/nlp_case_study/.venv/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


  1231 tok/s ±87  peak=25.3 MB
Benchmarking en stanza:en ...
  488 tok/s ±26  peak=13.6 MB
Benchmarking ru spacy:ru_core_news_lg ...
  12435 tok/s ±159  peak=70.9 MB
Benchmarking ru stanza:ru ...
  729 tok/s ±10  peak=9.3 MB


In [4]:
df = pd.DataFrame(rows)
df.to_csv("../results/performance.csv", index=False)
print("Saved results/performance.csv")
df

Saved results/performance.csv


,lang,parser,family,tokens_per_sec_mean,tokens_per_sec_std,peak_mb
0,en,spacy:en_core_web_trf,transition,1230.9,87.3,25.3
1,en,stanza:en,graph,488.4,26.3,13.6
2,ru,spacy:ru_core_news_lg,transition,12435.1,159.2,70.9
3,ru,stanza:ru,graph,729.4,10.3,9.3


## What we learned (from this run)

Numbers written to `results/performance.csv`:

| Lang | Parser (family) | tok/sec (mean ± std) | peak mem |
|---|---|---|---|
| EN | spaCy (transition) | 1,231 ± 87 | 25.3 MB |
| EN | Stanza (graph) | 488 ± 26 | 13.6 MB |
| RU | spaCy (transition) | **12,435 ± 159** | 70.9 MB |
| RU | Stanza (graph) | 729 ± 10 | **9.3 MB** |

- **Transition-based wins throughput by 2.5× on EN and ~17× on RU.** The RU gap is huge because spaCy RU uses a small CNN backbone (`ru_core_news_lg`) while EN uses a RoBERTa-based transformer (`en_core_web_trf`) — the EN throughput is capped by the transformer, not by the parser family.
- **Graph-based uses less peak memory in both languages** — opposite of what the "biaffine stores O(n²) scores" intuition suggests. The dominating allocator is the tagger/embedder, not the parser: spaCy's larger embedding matrices outweigh Stanza's attention tables in our regime. Memory only becomes an issue at larger batch sizes (not varied here).
- **Speed variance is small** across 3 repeats (std < 8% of mean everywhere) — the throughput numbers are reliable for the orders-of-magnitude comparison the poster draws.
